# Self-Led Support SXO Report

Reproducible working notebook for `docs/REPORT1.md`.

This notebook keeps the analysis scoped to the self-led support cohort defined in `sxo-strategy-proposal.md`:

- `/get-help/support-toolkit*`
- `/get-help/national-services*`
- `/get-help/hear-from-others*`

Primary warehouse sources:

- `searchconsole.seo_page_daily`
- `searchconsole.curated_search_query_page_daily`

External review:

- live page fetches from `lifeline.org.au`
- SerpAPI AU SERP checks
- `pytrends` for lightweight AU topic direction

In [ ]:
# Setup
from __future__ import annotations

import sys
import os
import re

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q 'plotly>=6.1.1' 'kaleido>=1.2.0'
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests
from google.cloud import bigquery
from lxml import html
from pytrends.request import TrendReq

import lifeline_theme
import lla_data.config as config
from lla_data.bq import get_client, load_sql_template, run_query

pd.set_option("display.max_colwidth", 140)
client = get_client()
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

PROJECT_ID = config.PROJECT_ID
SEARCHCONSOLE_DATASET = config.SEARCHCONSOLE_DATASET
SITE_BASE_URL = config.SITE_BASE_URL.rstrip("/")


NameError: name 'sys' is not defined

In [ ]:
# Parameters
ANALYSIS_END_DATE = date.today() - timedelta(days=4)
ANALYSIS_START_DATE = ANALYSIS_END_DATE - timedelta(days=89)
PAGE_PREFIXES = [
    "/get-help/support-toolkit",
    "/get-help/national-services",
    "/get-help/hear-from-others",
]
MIN_IMPRESSIONS = 500
TOP_N_CANDIDATES = 60
TOP_N_FINAL = 10
TOP_N_QUERIES = 8
MIN_QUERY_IMPRESSIONS = 100

print(f"Project: {PROJECT_ID}")
print(f"Search Console dataset: {SEARCHCONSOLE_DATASET}")
print(f"Analysis window: {ANALYSIS_START_DATE} to {ANALYSIS_END_DATE}")
print(f"Page prefixes: {PAGE_PREFIXES}")


In [ ]:
# Candidate pages from GA/GSC
candidate_sql = load_sql_template(
    "sql/notebooks/seo_self_led_support_candidate_pages.sql",
    project_id=PROJECT_ID,
    searchconsole_dataset=SEARCHCONSOLE_DATASET,
)
candidate_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", ANALYSIS_START_DATE),
    bigquery.ScalarQueryParameter("end_date", "DATE", ANALYSIS_END_DATE),
    bigquery.ArrayQueryParameter("page_prefixes", "STRING", PAGE_PREFIXES),
    bigquery.ScalarQueryParameter("min_impressions", "INT64", MIN_IMPRESSIONS),
    bigquery.ScalarQueryParameter("top_n", "INT64", TOP_N_CANDIDATES),
]
df_candidates = run_query(client, candidate_sql, params=candidate_params)
df_candidates.head(20)


In [ ]:
# Query drivers for shortlisted pages
shortlist_pages = df_candidates["page_path"].head(TOP_N_FINAL).tolist()
query_sql = load_sql_template(
    "sql/notebooks/seo_self_led_support_query_drivers.sql",
    project_id=PROJECT_ID,
    searchconsole_dataset=SEARCHCONSOLE_DATASET,
)
query_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", ANALYSIS_START_DATE),
    bigquery.ScalarQueryParameter("end_date", "DATE", ANALYSIS_END_DATE),
    bigquery.ArrayQueryParameter("pages", "STRING", shortlist_pages),
    bigquery.ScalarQueryParameter("min_query_impressions", "INT64", MIN_QUERY_IMPRESSIONS),
    bigquery.ScalarQueryParameter("top_n_queries", "INT64", TOP_N_QUERIES),
]
df_queries = run_query(client, query_sql, params=query_params)
df_queries.head(30)


In [ ]:
# Live page, SERP, and pytrends review helpers
def clean_text(value: str | None) -> str:
    return re.sub(r"\\s+", " ", value or "").strip()

def review_page(page_path: str, query: str) -> dict[str, object]:
    url = f"{SITE_BASE_URL}{page_path}"
    page_response = session.get(url, timeout=30)
    page_response.raise_for_status()
    root = html.fromstring(page_response.text)
    article = root.xpath("//article")
    article = article[0] if article else root

    body_text = " ".join(
        clean_text(p.text_content())
        for p in article.xpath(".//p")
        if clean_text(p.text_content())
    )

    result = {
        "page_path": page_path,
        "query": query,
        "title": clean_text(root.xpath("string(//title)")),
        "meta_description": clean_text("".join(root.xpath("//meta[@name='description']/@content"))),
        "h1": clean_text(article.xpath("string(.//h1)")),
        "headings": [
            clean_text(node.text_content())
            for node in article.xpath(".//*[self::h2 or self::h3]")
            if clean_text(node.text_content())
        ][:10],
        "has_support_cta": any(token in body_text.lower() for token in ["13 11 14", "131114", "chat with us", "text us"]),
    }

    serp_api_key = os.getenv("SERP_API_KEY")
    if serp_api_key:
        serp_response = session.get(
            "https://serpapi.com/search.json",
            params={
                "engine": "google",
                "q": query,
                "gl": "au",
                "hl": "en",
                "num": 10,
                "api_key": serp_api_key,
            },
            timeout=30,
        )
        serp_response.raise_for_status()
        serp_payload = serp_response.json()
        organic = serp_payload.get("organic_results", [])[:5]
        result["serp_lifeline_rank"] = next(
            (
                idx + 1
                for idx, row in enumerate(organic)
                if "lifeline.org.au" in (row.get("link") or "")
            ),
            None,
        )
        result["serp_top_domains"] = [row.get("displayed_link") or row.get("link") for row in organic]
        result["serp_has_related_questions"] = bool(serp_payload.get("related_questions"))

    try:
        pytrends = TrendReq(hl="en-AU", tz=600)
        pytrends.build_payload([query], timeframe="today 12-m", geo="AU")
        trend = pytrends.interest_over_time()
        if not trend.empty:
            values = trend[query].tolist()
            halfway = len(values) // 2 or 1
            result["trend_first_half_avg"] = sum(values[:halfway]) / len(values[:halfway])
            result["trend_second_half_avg"] = sum(values[halfway:]) / len(values[halfway:])
            result["trend_latest_complete"] = values[-2] if len(values) > 1 else values[-1]
    except Exception as exc:
        result["trend_error"] = str(exc)

    return result


## Scoring notes

Use the report rubric for the final ranking:

- `35%` SEO upside
- `35%` mission and user-value fit
- `20%` implementation practicality
- `10%` measurement quality

Keep `docs/REPORT1.md` as the final narrative output. The notebook's job is to make the underlying evidence easy to rerun and inspect.